<a href="https://colab.research.google.com/github/islamsamian22-ship-it/Python-prohects/blob/main/Energy_delta_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred) ** 0.5

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. Skipping XGBoost model.")
df = pd.read_csv("/content/drive/MyDrive/solar_weather.csv")

target = "Energy delta[Wh]"

print("Shape:", df.shape)
print(df.head())

missing = df.isnull().sum()
print("\nMissing values per column:\n", missing[missing > 0])
df = df.fillna(df.median(numeric_only=True))

df["Time"] = pd.to_datetime(df["Time"])
df["hour_of_day"] = df["Time"].dt.hour
df["day_of_year"] = df["Time"].dt.dayofyear
df["GHI_clouds_interaction"] = df["GHI"] * (1 - df["clouds_all"] / 100)
df["is_precipitating"] = ((df["rain_1h"] > 0) | (df["snow_1h"] > 0)).astype(int)

numeric_df = df.select_dtypes(include="number")
corr = numeric_df.corr()

# GHI is the strongest by far (~0.91), which makes sense: it's the direct measure of sunlight hitting the panels.
# If two features turn out highly correlated WITH EACH OTHER (not the target),it's worth dropping one - keeping both just adds noise and makes linear
# models unstable without adding new information.

plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png")
plt.close()

target_corr = corr[target].drop(target).sort_values(key=abs, ascending=False)
print("\nTop correlations with target:\n", target_corr.head(10))


plt.figure(figsize=(8, 5))
sns.scatterplot(x="clouds_all", y=target, data=df, alpha=0.2)
plt.title("Cloud Cover vs Energy Generation")
plt.xlabel("Cloud Cover (%)")
plt.ylabel("Energy delta [Wh]")
plt.tight_layout()
plt.savefig("clouds_vs_energy.png")
plt.close()

# clouds_all comes out negatively correlated (~ -0.19), confirming more cloud = less energy, as expected.

plt.figure(figsize=(6, 5))
sns.boxplot(y=df[target])
plt.title("Energy Output - Outlier Check")
plt.tight_layout()
plt.savefig("energy_boxplot.png")
plt.close()

Q1 = df[target].quantile(0.25)
Q3 = df[target].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = df[(df[target] < lower) | (df[target] > upper)]
print(f"\nOutliers detected (IQR method): {len(outliers)} rows")

# Note: a lot of these "outliers" are just night-time zeros pulling the
# quartiles down - not bad data, just how a zero-inflated variable looks.

drop_cols = ["Time", target]
X = df.drop(columns=drop_cols)
X = X.select_dtypes(include="number")
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

models = {"Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
if XGB_AVAILABLE:
    models["XGBoost"] = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42)

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2})
    print(f"\n{name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.4f}")

results_df = pd.DataFrame(results).sort_values("RMSE")
print("\nFinal comparison:\n", results_df)

import joblib
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
joblib.dump(best_model, "best_solar_model.pkl")
print(f"\nBest model ({best_model_name}) saved as best_solar_model.pkl")

# Tree models (RF/XGBoost) usually beat Linear Regression here because output isn't a straight line vs weather - it's flat at zero all night,then ramps up non-linearly, with clouds mattering more or less depending
# on GHI. Trees can capture that; a single linear line can't.
# XGBoost trade-off: needs more tuning, easier to overfit if not careful,
# and less interpretable than one plain decision tree.

import joblib
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
joblib.dump(best_model, "best_solar_model.pkl")
print(f"\nBest model ({best_model_name}) saved as best_solar_model.pkl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (196776, 17)
                  Time  Energy delta[Wh]  GHI  temp  pressure  humidity  \
0  2017-01-01 00:00:00                 0  0.0   1.6      1021       100   
1  2017-01-01 00:15:00                 0  0.0   1.6      1021       100   
2  2017-01-01 00:30:00                 0  0.0   1.6      1021       100   
3  2017-01-01 00:45:00                 0  0.0   1.6      1021       100   
4  2017-01-01 01:00:00                 0  0.0   1.7      1020       100   

   wind_speed  rain_1h  snow_1h  clouds_all  isSun  sunlightTime  dayLength  \
0         4.9      0.0      0.0         100      0             0        450   
1         4.9      0.0      0.0         100      0             0        450   
2         4.9      0.0      0.0         100      0             0        450   
3         4.9      0.0      0.0         100      0             0        450   
4    